<a href="https://colab.research.google.com/github/moodyastra/Quickstart_Pymol/blob/main/Malachi_Moody_Exercise1_Play_with_pymol_and_biotite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pymol & Biotite


### How to do this excercise:

* Step1: copy this colab file to your own google drive by `File` -> `Save a Copy to Drive`;  
* Step2: Open the notebook in your own drive;  
* Step3: To run the code block, press `Enter+Ctrl` to run it. Run the notebook top‑to‑bottom.  


### What to expect?
- **Who**: This is intended for folks who has no computational or biochemical background.
- **Aim**: biomolecules are pretty, we can use `pymol` to check them and use `biotite` to change them. It is super easy to write simple python script do make changes.
- **What**: This is a jupyter notebook that built on google colab system (aka you will get a computer node from google). Linna wrote some simple codes, to just give you a quick taste of what may you learn at the An lab.

### What does this excercise do?

0. Learn to use google colab and jupyter notebook system
1. Installs **PyMOL‑open‑source** & **Biotite**  
2. Fetches a pdb from PDB database. Example: PDB 8VEJ.
3. Reads structure into PyMOL for visualisation  
4. Extracts *chain A* sequence with Biotite  

### How to read a jupyter notebook?
- `# everything after # xxxx` is comments, it explains the codes and what each step means
- Don't worry if you don't understand each line, try to figure out, what does each code block is trying to do.
- When codes are running, you will see log print out below the code blocks.

### Additional notes
- Google colab embeded Gemini, you can ask questions
- Google colab enabled autocompletion, use Tab to allow autocompletion

In [ ]:
# @title 🛠️ Install dependencies (takes ~1‑2 min)
!apt-get update -y        # -qq keeps the log short
!apt-get install -y pymol # open-source PyMOL 2.x
!pip -q install biotite py3Dmol
!pip install icecream # this is a package commonly used for debugging

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [4,685 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [113 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,706 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,191 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,287 kB]

In [ ]:
# import python packages needed
from biotite.database.rcsb import fetch
from biotite.structure.io.pdb import PDBFile
import biotite
import os
from icecream import ic
from biotite.sequence import ProteinSequence
import numpy as np

# modules which allows visualization
import io
import py3Dmol
from IPython.display import display

# if functions imported correct, there should be no error

In [ ]:
# 📥 Fetch PDB 8VEJ and save locally
pdb_file = fetch("8VEJ", "pdb", os.getcwd())
print("Downloaded:", pdb_file)
# your google directory should now have the downloaded pdb


Downloaded: /content/8VEJ.pdb


what is a pdb file? [PDB101](https://pdb101.rcsb.org/learn/guide-to-understanding-pdb-data/pdb-overview); [information of each column](https://pdb101.rcsb.org/learn/guide-to-understanding-pdb-data/dealing-with-coordinates)

In [ ]:
pdb = PDBFile.read(pdb_file)
atom_array = pdb.get_structure(model=1) # read the first model
atom_array[:10] # this is the pdb file you read. Take a look at it, what are the information there? what does [:10] mean? What would happen if you remove [:10]?


# Thoughts: [:10] must be an array restricter that only returns the elements from the main array with 0 being an implied starting index [0,10) (python slicing).
# I also notice that not all characteristics of all atoms are printed in the Collab workspace, instead "..." appears. Is there a way to see all the elements without creating a for loop?
# Putting in another command like "print(len(atom_array))" (aside: also hovering over the array seems to give the same value, called "shape") seems to remove any array outputs shown before. Is there a reason for that?
# "np.array" seems to be a Numpy command and the array seems to print general information about the specific atom. I understand most attributes are just identifiers, but why is "ins_code" blank?
# And what does it mean for "hetero" to be False?

array([
	Atom(np.array([ 20.262, -14.827, -10.769], dtype=float32), chain_id="A", res_id=4, ins_code="", res_name="SER", hetero=False, atom_name="C", element="C"),
])

In [ ]:
# ------------------------------------------------------------------
# assume you already have `atom_array` (e.g. from PDBFile.read(...))
# ------------------------------------------------------------------

# 1️⃣  Convert AtomArray → PDB text in memory
buffer = io.StringIO()
pdb_out = PDBFile()
pdb_out.set_structure(atom_array)   # the whole structure, or slice if you like
pdb_out.write(buffer) # Is an IOString a file?
pdb_text = buffer.getvalue()

# 2️⃣  Push that PDB string into py3Dmol and show a cartoon
view = py3Dmol.view(width=500, height=450)
view.addModel(pdb_text, 'pdb')      # just pass the string
view.setStyle({'cartoon': {}}) # Q: How many different styles are there?
view.zoomTo() # Remiscence of Zoom_Stat on a TI-84
view.show()

# (In notebooks, `view.show()` is optional; `view` will auto-display.)
# in this pdb, on chain B, there is a ligand, can you show it in sticks in the figure?
# ligand (hetero atoms) on chain B as sticks, coloured by element

# example code. Remove the # to run it.
view.addStyle({'chain': 'B', 'hetflag': True},
               {'stick': {'radius': 37.8, 'colorscheme': 'element'}})

view.zoomTo()
view.show()

# Additional Thoughts: Original question now answered ligands are defined as hetero atoms. However, are there any additional hetero atoms? And why do hetero atoms not appear normally?
# "Sticks" seem to be some other form of representation, but I don't understand how the red endpoints of the ligand are chosen? Is there a specific reason they are where they are?
# I see that it says "'colorscheme':'element'" does this mean every element has its own colorscheme? What color do new elements or radioactive elements appear as?
# Moreover, I see that it sets the radius of the coloring to ".25". Is there a measure of scale for this value (or just comparative to 450 and 500)?

# in the figure below, you can use mouse to play around the protein.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 🧬 Extract sequence of chain A
mask = (atom_array.atom_name == 'CA') & (atom_array.chain_id == 'A') # CA = Alpha Carbon? & Mask = a specialized slicer array
res_ids = atom_array.res_id[mask] # Gets the residue ids from the array of atoms left after the mask
uniq_idx = np.sort(np.unique(res_ids, return_index=True)[1]) # Sorts the indexes of all unique residue ids after the mask in ascending order

res_3letter = atom_array.res_name[mask][uniq_idx] # Gets the residue names from the masked list by using the indices from uniq_idx

seq = ''.join(ProteinSequence(res_3letter.tolist())) # Squishes all 3-letter residues into a side-by-side one letter list

print("Chain A length:", len(seq))
print(seq)
print(f"the 6th aa: {seq[5]}") # get the 6th residue, number starts from 0

Chain A length: 240
PSAEIFEDFLDWYEEIIESTDKRVEAIVEELGGELTEEMKEALAMARAAMAAFEKEGDVFRLASVGATAALMVWFHLRNIPLAGIARAWLLMLAQLEAGADVDTALAAAAEVLREYGVSEELVAAAAAEARALVESLTPFELASLAALAAARLLFTARNNPLAKIIQAALDLYNRTVDMSTEEAVEAALEAMKELGASEESLERLKELVESARKRGIELTPFEVALLAYYVLVLDYLKKS
the 6th aa: F


In [ ]:
atom_array[(atom_array.res_id==6) & (atom_array.atom_name == 'CA') & (atom_array.chain_id == 'A')] # what does this line mean? Then check above, did it return the correct residue name? If not, why?

#This line seems to return an array with the attributes of the alpha carbon atom in chain A with a residue id of 6, which in this PDB appears to be Glutamic Acid. It doesn't return the same residue name
# as the sixth unique residue as it is not necessarily the res_id = 6 in this PBD. res_id = 6 explicity selects the prearranged residue in the PBD not the 6th overall ordered element of alpha carbons in chain A.

array([
	Atom(np.array([ 19.136, -17.04 , -14.321], dtype=float32), chain_id="A", res_id=6, ins_code="", res_name="GLU", hetero=False, atom_name="CA", element="C"),
])

In [29]:
atom_array[(atom_array.res_id==2) & (atom_array.atom_name == 'CA') & (atom_array.chain_id == 'A')] # this AtomArray is empty
## Answer: if you check the file, you will realize, the pdb file does not have the first 2 residues, which caused error in getting the correct sequence starting from residue 1.
# Thoughts: I believe this is due to the incompleteness of the PDB archive as stated in PDB101: "Also, many PDB entries are missing portions of the molecule that were not observed in the experiment."
# But I am curious, is there a reason why an author would choose to make certain indexes for residue ids blank like 1 & 2 for this file?

array([
])

*Nice! You are using computational codes to work on biomolecules!
Why using codes?
If you are only working on one pdb, it would be significantly easier if you just edit and play by hands.
But...what if you have 100,000 pdbs? That's a common scale for protein engineering in academia and industry. Then you have to use computer to make things faster.*

---------Notes from Linna-----------------------------------------------------

At the An lab, you will routinly use `python` and `jupyter notebook` (like what you're using here. I think this will be the future for all protein engineers, they will become our routine toolkits, just like other routine biochemical toolkits, such as pH meter, or a pipette. Never learnt anything like this before? Don't worry, you will be fine.

If you worked with codes before, you probably find this notebook basic, congratulations, you are more than ready to work in the An lab. Now you will need to learn some biochemistry to know how your skill will shift our health industry.
